# Analyzing Mobile App Data for App Store and Google Play Markets

In this project, we analyze data from the [Google Play Store](https://dq-content.s3.amazonaws.com/350/googleplaystore.csv) and [Apple App Store](https://dq-content.s3.amazonaws.com/350/AppleStore.csv) to explore patterns in mobile applications. As data analysts working for a company that develops apps for both platforms, we focus on understanding how different app characteristics; such as genre, ratings, and pricing, relate to user engagement.

The goal of this analysis is to identify the types of apps that are most likely to attract a large number of users. Since the company’s revenue is driven by in-app advertising, higher user engagement directly translates to higher revenue. The insights from this project will help guide developers in building apps that maximize reach and user interaction.

**Step 1: Open the two data sets (App store data and play store data)**

In [1]:
from csv import reader
import pandas as pd
import numpy as np

def read_csv(file_name):
     with open(file_name, encoding = 'utf-8') as file:
         return list(reader(file))

apple_store_data = read_csv('AppleStore.csv') #app store data
google_play_store_data = read_csv('googleplaystore.csv') #play store data

#Convert data to df for quick analysis
df_apple = pd.DataFrame(apple_store_data)
df_google = pd.DataFrame(google_play_store_data)

**Step 2: Explore the two data sets:**

    1. Write a function to repeatedly explore rows in a more readable way
    2. The function also shows the number of rows and columns for any given data set

In [2]:
def explore_data(dataset, start, end, rows_and_columns=False):
    dataset_slice = dataset[start:end]    
    for row in dataset_slice:
        print(row)
        print('\n')

    if rows_and_columns:
        print('Number of rows:', len(dataset))
        print('Number of columns:', len(dataset[0]))
    
#rows_and_columns=False makes the argument optional, so that we can call the function in two ways:
    #explore_data(dataset, 0, 5)
    #explore_data(data, 0, 5, True)

In [3]:
apple_dataset = apple_store_data[1:]
google_dataset = google_play_store_data[1:]

#Print the first few rows of each dataset and the number of rows and columns 
print(explore_data(apple_dataset, 0, 5, True))
print(explore_data(google_dataset, 0, 5, True))

#Print column names
print('App store columns:')
print(apple_store_data[0])
print('\n')
print('Play store columns:')
print(google_play_store_data[0])


['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1']


['389801252', 'Instagram', '113954816', 'USD', '0.0', '2161558', '1289', '4.5', '4.0', '10.23', '12+', 'Photo & Video', '37', '0', '29', '1']


['529479190', 'Clash of Clans', '116476928', 'USD', '0.0', '2130805', '579', '4.5', '4.5', '9.24.12', '9+', 'Games', '38', '5', '18', '1']


['420009108', 'Temple Run', '65921024', 'USD', '0.0', '1724546', '3842', '4.5', '4.0', '1.6.2', '9+', 'Games', '40', '5', '1', '1']


['284035177', 'Pandora - Music & Radio', '130242560', 'USD', '0.0', '1126879', '3594', '4.0', '4.5', '8.4.1', '12+', 'Music', '37', '4', '1', '1']


Number of rows: 7197
Number of columns: 16
None
['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


['Coloring book moana', 'ART_AND_DESIGN', '3.9', '96

**Step 3: Data cleaning**

    1. Detect inaccurate data and correct/ remove it
    2. Detect and remove duplicates
    3. Remove non-English apps
    4. Remove paid apps

From the [discussion](https://www.kaggle.com/datasets/lava18/google-play-store-apps/discussion) forum, it is [mentioned](https://www.kaggle.com/datasets/lava18/google-play-store-apps/discussion/66015) that row 10472 has a missing 'Rating' and a column shift occurs for the next columns 

In [4]:
error_row = google_dataset[10472]
print(error_row)
print(len(error_row)) #confirms there is a column shift (12 instead of 13)

#remove the row
del google_dataset[10472]

print(len(google_dataset))

['Life Made WI-Fi Touchscreen Photo Frame', '1.9', '19', '3.0M', '1,000+', 'Free', '0', 'Everyone', '', 'February 11, 2018', '1.0.19', '4.0 and up']
12
10840


In [5]:
#Remove duplicates

#define a function to find duplicates in any dataset
def duplicate_finder(dataset):
    duplicate_apps = []
    unique_apps = []
    for row in dataset:
        name = row[0]
        if name in unique_apps:
            duplicate_apps.append(name)
        else:
            unique_apps.append(name)
    return duplicate_apps
    
google_duplicates = duplicate_finder(google_dataset)

print('Number of Google Duplicate Apps:', len(google_duplicates))
print('Examples of duplicate apps:', google_duplicates[:5])

Number of Google Duplicate Apps: 1181
Examples of duplicate apps: ['Quick PDF Scanner + OCR FREE', 'Box', 'Google My Business', 'ZOOM Cloud Meetings', 'join.me - Simple Meetings']


Need to remove  duplicate entries and keep only one entry per app.
The main difference happens on the fourth position of each row, which corresponds to the number of reviews. 
The different numbers show the data was collected at different times.
The higher the number of reviews, the more recent the data is.
Keep the rows that have the highest number of reviews. 

1. Create a dictionary where each key is a unique app name, and the value is the highest number of reviews of that app
2. Use the dictionary to create a new data set, which will have only one entry per app (and we only select the apps with the highest number of reviews)


In [6]:
reviews_max = {}

for app in google_dataset:
    name = app[0]
    n_reviews = float(app[3])
    if name in reviews_max and reviews_max[name] < n_reviews:
        reviews_max[name] = n_reviews
    elif name not in reviews_max:
        reviews_max[name] = n_reviews

In [7]:
print('Expected length:', len(google_dataset) - 1181)
print('Actual length:', len(reviews_max))

Expected length: 9659
Actual length: 9659


In [8]:
#Use the reviews_max dictionary to remove the duplicates
google_clean = []
already_added = []
for row in google_dataset:
    name = row[0]
    n_reviews = float(row[3])

    if n_reviews == reviews_max[name] and name not in already_added:
        google_clean.append(row)
        already_added.append(name)
    

In [9]:
explore_data(google_clean, 0, 3, True)

['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'ART_AND_DESIGN', '4.7', '87510', '8.7M', '5,000,000+', 'Free', '0', 'Everyone', 'Art & Design', 'August 1, 2018', '1.2.4', '4.0.3 and up']


['Sketch - Draw & Paint', 'ART_AND_DESIGN', '4.5', '215644', '25M', '50,000,000+', 'Free', '0', 'Teen', 'Art & Design', 'June 8, 2018', 'Varies with device', '4.2 and up']


Number of rows: 9659
Number of columns: 13


In [10]:
#Check if apple data has any duplicates:
#We can use duplicate_finder function
    
apple_duplicates = duplicate_finder(apple_dataset)

print('Number of Apple Duplicate Apps:', len(apple_duplicates))
print('Examples of duplicate apps:', apple_duplicates[:5])

Number of Apple Duplicate Apps: 0
Examples of duplicate apps: []


# Removing Non-English Apps
Both datasets have apps with non-english names.

We're not interested in keeping these apps, so we'll remove them:

1. Remove apps with a name containing a symbol that isn't commonly used in English text (English text  includes letters from the English alphabet, numbers composed of digits from 0 to 9, punctuation marks (., !, ?, ;), and other symbols (+, *, /))
   * All characters specific to English texts are encoded using the ASCII (American Standard Code for Information Interchange) standard.
   * Each ASCII character has a corresponding number between 0 and 127 associated with it.
   * We can take advantage of that to build a function that checks an app name and tells us whether it contains non-ASCII characters.
   * If the number is equal to or less than 127, then the character belongs to the set of common English characters.
   * If an app name contains a character that is greater than 127, then it probably means that the app has a non-English name.
   * We built this function below, and we use the built-in ord() function to find out the corresponding encoding number of each character.

In [11]:
#Since strings are indexable and iterable, 
#we can use indexing to select an individual character and also iterate on the string using a for loop
string = 'abcde'
for character in string:
    print(ord(character))

97
98
99
100
101


In [12]:
#function to find the non-ascii characters
def english_app_finder(app_name):
    for character in app_name:
        if ord(character) > 127:
            return False

    return True #We put the return statement outside the if loop to check the entire string. 
    #If we put it within the if loop, it will never check the rest of the string's characters

In [13]:
#Test the function:
print(english_app_finder('Instagram'))
print(english_app_finder('爱奇艺PPS -《欢乐颂2》电视剧热播'))
print(english_app_finder('Docs To Go™ Free Office Suite'))
print(english_app_finder('Instachat 😜'))

True
False
False
False


The above function doesn't correctly identify certain English app names like 'Docs To Go™ Free Office Suite' and 'Instachat 😜'. 
This is because emojis and characters like ™ fall outside the ASCII range and have corresponding numbers over 127.
We'll lose useful data since many English apps will be incorrectly labeled as non-English. 
To minimize the impact of data loss, we'll only remove an app **if its name has more than three characters with corresponding numbers falling outside the ASCII range.**

In [14]:
#New modified function:
def english_app_finder(app_name):
    non_ascii_characters = 0
    
    for character in app_name:
        if ord(character) > 127:
            non_ascii_characters += 1
    
    if non_ascii_characters > 3:
        return False
    else:
        return True 

In [15]:
#test function
print(english_app_finder('Instagram'))
print(english_app_finder('爱奇艺PPS -《欢乐颂2》电视剧热播'))
print(english_app_finder('Docs To Go™ Free Office Suite'))
print(english_app_finder('Instachat 😜'))

True
False
True
True


## Use the new function to filter out non-English apps from both datasets. 

In [16]:
#Loop through each dataset. If an app name is identified as English, append the whole row to a separate list. 
english_google_clean = []
english_apple_dataset = []

for row in google_clean:
    app_name = row[0]
    if english_app_finder(app_name):
        english_google_clean.append(row)

explore_data(english_google_clean, 0, 3, True)


for row in apple_dataset:
    app_name = row[1]
    if english_app_finder(app_name):
        english_apple_dataset.append(row)

explore_data(english_apple_dataset, 0, 3, True)

['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'ART_AND_DESIGN', '4.7', '87510', '8.7M', '5,000,000+', 'Free', '0', 'Everyone', 'Art & Design', 'August 1, 2018', '1.2.4', '4.0.3 and up']


['Sketch - Draw & Paint', 'ART_AND_DESIGN', '4.5', '215644', '25M', '50,000,000+', 'Free', '0', 'Teen', 'Art & Design', 'June 8, 2018', 'Varies with device', '4.2 and up']


Number of rows: 9614
Number of columns: 13
['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1']


['389801252', 'Instagram', '113954816', 'USD', '0.0', '2161558', '1289', '4.5', '4.0', '10.23', '12+', 'Photo & Video', '37', '0', '29', '1']


['529479190', 'Clash of Clans', '116476928', 'USD', '0.0', '2130805', '579', '4.5', '4.5', '9.24.12', '9+', 'G

## Isolating the Free Apps

* We are only interested in apps that are free to download and install.
* Our datasets contain both free and non-free apps
* We'll need to isolate only the free apps for our analysis.

In [17]:
#Loop through each dataset to isolate the free apps in separate lists
#Identify relevant columns. Need column index 7 for google and 4 for appstore

free_apple_store = []
free_google = []

for row in english_apple_dataset:
    price = row[4]
    if price == '0.0':
        free_apple_store.append(row)

for row in english_google_clean:
    price = row[7]
    if price == '0':
        free_google.append(row)

print(len(free_apple_store))        
print(len(free_google))

3222
8864


## App Analysis

### Most Common Apps by Genre: Part One
* The end goal is to add the app on both Google Play and the App Store
* We need to find app profiles that are successful in both markets
* For instance, a profile that works well for both markets might be a productivity app that makes use of gamification.

We begin the analysis by determining the most common genres for each market. 

For this, we'll need to build frequency tables for a few columns in our datasets.

In [18]:
#Identify columns we could use to generate frequency tables
print('Apple Data:', df_apple.head())
print('Google Data:', df_google.head())

Apple Data:           0               1           2         3      4                 5   \
0         id      track_name  size_bytes  currency  price  rating_count_tot   
1  284882215        Facebook   389879808       USD    0.0           2974676   
2  389801252       Instagram   113954816       USD    0.0           2161558   
3  529479190  Clash of Clans   116476928       USD    0.0           2130805   
4  420009108      Temple Run    65921024       USD    0.0           1724546   

                 6            7                8        9            10  \
0  rating_count_ver  user_rating  user_rating_ver      ver  cont_rating   
1               212          3.5              3.5     95.0           4+   
2              1289          4.5              4.0    10.23          12+   
3               579          4.5              4.5  9.24.12           9+   
4              3842          4.5              4.0    1.6.2           9+   

                  11               12               13        

### Most Common Apps by Genre: Part Two

We'll build a frequency table for the prime_genre column of the Apple Store data set, and the Genres and Category columns of the Google Play data set.

We'll build 2 functions:
* One to generate frequency tables and show percentages using dictionaries
* Another to display percentages in descending order since dictionaries don't have order (display_table() function):
  1. Takes in two parameters: dataset and index. dataset is a list of lists, and index is an integer
  2. Generates a frequency table using the freq_table() function


In [19]:
#frequency_table
#We're building a frequency table for prime_genre column of Apple store data and Genres and Category columns of google data
#For each of these columns, we need to know the percentages associated with each unique entry
def frequency_table(dataset, index):
    table = {} 
    total = 0 #for calculating percentage
    
    for row in dataset:
        total += 1 #Total number of all rows, whether unique or not
        value = row[index] #Acess genre in specific cells

        #Dictionary 1: {genre : count}
        if value in table:
            table[value] += 1 #Increase the count by one if I have the genre from the specific cell in my dictionary already
        else:
            table[value] = 1 #Make the first record if the value is not already present
    
    table_percentages = {}
    for key in table: #For each genre, calculate the percentage
        percentage = (table[key] / total) * 100
        table_percentages[key] = percentage 
    
    return table_percentages

#display_table
def display_table(dataset, index):
    table = frequency_table(dataset, index)
    table_display = []
    for key in table:
        #To ensure the sorting works right, the dictionary value comes first (table[key]), and the dictionary key (key) comes second
        key_val_as_tuple = (table[key], key)
        table_display.append(key_val_as_tuple)
        
    table_sorted = sorted(table_display, reverse = True)
    for entry in table_sorted:
        print(entry[1], ':', entry[0])

In [21]:
#display_table(english_apple_dataset, 11)
#display_table(english_google_clean, 1)
#display_table(english_google_clean, 9)

### Most Common Apps by Genre: Part Three
Focus: Analyzing the frequency tables

#### Apple store data:
* More than half (54.86%) are games.
* Entertainment apps are close to 8%, followed by education, then photo and video apps.
* The general impression is that App Store (at least the part containing free English apps) is dominated by apps that are designed for fun (games, entertainment, photo and video, social networking, sports, music, etc.), while apps with practical purposes (shopping, utilities, productivity, lifestyle, etc.) are rarer.
* However, the fact that fun apps are the most numerous doesn't also imply that they also have the greatest number of users — the demand might not be the same as the offer.


#### Google Data:
* The landscape seems significantly different on Google Play.
* There are not that many apps designed for fun, and it seems that a good number of apps are designed for practical purposes (family, tools, business, lifestyle, productivity, etc.).
* The difference between the Genres and the Category columns is not crystal clear, but one thing we can notice is that the Genres column is much more granular (it has more categories).
* We're only looking for the bigger picture at the moment, so we'll only work with the Category column moving forward.

### Most Popular Apps by Genre on the App Store
* The frequency tables showed us that apps designed for fun dominate the App Store, while Google Play shows a more balanced landscape of both practical and fun apps.
* Now, we'd like to determine the kind of apps with the most users (what genres are the most popular).
* We can calculate the average number of installs for each app genre.
* For the Google data set, we can find this information in the **Installs** column
* However, this information is missing for the Apple data set.
* As a workaround, we'll use the total number of user ratings, which we can find in the **rating_count_tot** column.

##### Average number of user ratings per app genre on the App Store (apple dataset)
1. Isolate the apps of each genre
2. Add up the user ratings for the apps of that genre
3. Divide the sum by the number of apps belonging to that genre (not by the total number of apps)

In [49]:
#Get genres frequency table
apple_genres = frequency_table(english_apple_dataset, 11)
print(apple_genres)

for genre in apple_genres: #extracts the keys only from the dictionary
    total_genre_rating = 0
    len_genre = 0
    for app in english_apple_dataset:
        app_genre = app[11]
        if app_genre == genre:
            app_rating = float(app[5])
            total_genre_rating += app_rating
            len_genre += 1
    
    avg_genre_rating = total_genre_rating / len_genre
    print(genre, ':', avg_genre_rating)

{'Social Networking': 2.037845705967977, 'Photo & Video': 5.515122109008572, 'Games': 54.860100274947435, 'Music': 2.215752870774705, 'Reference': 0.8571890667960537, 'Health & Fitness': 2.6686074721009216, 'Weather': 1.1159631246967492, 'Utilities': 3.4449296458030085, 'Travel': 0.9704027171276078, 'Shopping': 1.3747371825974446, 'News': 0.9218825812712276, 'Navigation': 0.452854601326217, 'Lifestyle': 1.6011644832605532, 'Entertainment': 7.261846999838266, 'Food & Drink': 0.7116286592269124, 'Sports': 1.6820313763545207, 'Book': 0.8895358240336406, 'Finance': 0.7924955523208799, 'Education': 6.6310852337053205, 'Productivity': 2.7171276079573023, 'Business': 0.8571890667960537, 'Catalogs': 0.08086689309396733, 'Medical': 0.3396409509946628}
Social Networking : 60253.84920634921
Photo & Video : 14688.715542521993
Games : 15586.759433962265
Music : 29047.109489051094
Reference : 27037.188679245282
Health & Fitness : 10802.157575757576
Weather : 23145.246376811596
Utilities : 7927.52582

**Conclusion:** On average, navigation apps have the highest number of user reviews.

### Most Popular Apps by Genre on Google Play